# GALVAT Trial 7 Training, Evaluation, Threshold Selection, ROI FP, and ONNX Export

This notebook restores the Trial 7 640 training dataset from Drive, audits it, trains YOLO26s on A100, selects the operating point by **GALVAT weighted score**, evaluates background false positives and clutter/spoken burden, creates readable charts/tables, exports ONNX, and saves all outputs to Drive.

Do not tune thresholds on test. Validation chooses the operating point; test is run once at the selected setting.

In [ ]:
# Cell 1: Environment setup

from pathlib import Path
import os, json, yaml, shutil, zipfile, math, random, datetime, hashlib, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

from google.colab import drive

random.seed(42)
np.random.seed(42)

drive.mount('/content/drive')

# Install/upgrade only if needed. Keep quiet to reduce notebook noise.
try:
    import ultralytics
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxruntime'])

from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt
import seaborn as sns

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
assert torch.cuda.is_available(), 'Use an A100/GPU runtime before training.'

In [ ]:
# Cell 2: Restore Trial 7 640 training dataset zip from Drive

DATASET_ZIP = Path('/content/drive/MyDrive/galvat_trial7_training_datasets/galvat_trial7_training_dataset_640_2026-06-11_17-49-31.zip')
RESTORE_ROOT = Path('/content/galvat_trial7_training_restore')

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'

if RESTORE_ROOT.exists():
    shutil.rmtree(RESTORE_ROOT)
RESTORE_ROOT.mkdir(parents=True, exist_ok=True)

print('Unzipping:', DATASET_ZIP)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(RESTORE_ROOT)

data_yaml_candidates = list(RESTORE_ROOT.rglob('dataset/data.yaml'))
assert data_yaml_candidates, 'No dataset/data.yaml found in restored archive.'

DATA_YAML = data_yaml_candidates[0]
DATA_ROOT = DATA_YAML.parent

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

# Make paths explicit and local for Colab.
data_cfg['path'] = str(DATA_ROOT)
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
data_cfg['test'] = 'test/images'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

names_obj = data_cfg['names']
if isinstance(names_obj, dict):
    NAMES = [names_obj[i] if i in names_obj else names_obj[str(i)] for i in range(len(names_obj))]
else:
    NAMES = list(names_obj)

print('DATA_ROOT:', DATA_ROOT)
print('DATA_YAML:', DATA_YAML)
print('Class count:', len(NAMES))
for i, n in enumerate(NAMES):
    print(i, n)

In [ ]:
# Cell 3: Audit dataset before training

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def audit_yolo_dataset(data_root, names):
    split_rows = []
    class_rows = []
    box_counts = Counter()
    image_counts = Counter()
    empty_counts = Counter()
    bad_rows = []

    for split, img_subdir in [('train', 'train/images'), ('valid', 'valid/images'), ('test', 'test/images')]:
        img_dir = data_root / img_subdir
        lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
        img_files = [p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]

        for img_path in img_files:
            lbl_path = lbl_dir / f'{img_path.stem}.txt'
            lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
            seen = set()
            valid = 0
            for i, line in enumerate(lines):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append((split, img_path.name, i + 1, 'bad_column_count', line))
                    continue
                try:
                    cls_id = int(float(parts[0]))
                    x, y, w, h = [float(v) for v in parts[1:]]
                except Exception:
                    bad_rows.append((split, img_path.name, i + 1, 'parse_error', line))
                    continue
                if cls_id < 0 or cls_id >= len(names):
                    bad_rows.append((split, img_path.name, i + 1, 'bad_class_id', line))
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append((split, img_path.name, i + 1, 'bad_coords', line))
                    continue
                cls = names[cls_id]
                box_counts[(split, cls)] += 1
                seen.add(cls)
                valid += 1
            if valid == 0:
                empty_counts[split] += 1
            for cls in seen:
                image_counts[(split, cls)] += 1

        split_rows.append({'split': split, 'images': len(img_files), 'empty_label_images': empty_counts[split]})

    for cls in names:
        class_rows.append({
            'class': cls,
            'train_images': image_counts[('train', cls)],
            'valid_images': image_counts[('valid', cls)],
            'test_images': image_counts[('test', cls)],
            'total_images': sum(image_counts[(s, cls)] for s in ['train', 'valid', 'test']),
            'train_boxes': box_counts[('train', cls)],
            'valid_boxes': box_counts[('valid', cls)],
            'test_boxes': box_counts[('test', cls)],
            'total_boxes': sum(box_counts[(s, cls)] for s in ['train', 'valid', 'test']),
        })

    return pd.DataFrame(split_rows), pd.DataFrame(class_rows).sort_values('total_images'), pd.DataFrame(bad_rows, columns=['split','file','line','reason','text'])

split_summary, class_summary, bad_labels = audit_yolo_dataset(DATA_ROOT, NAMES)
display(split_summary)
display(class_summary)
print('Bad label rows:', len(bad_labels))
if len(bad_labels):
    display(bad_labels.head(50))

missing = class_summary[class_summary['total_images'] == 0]
low = class_summary[(class_summary['total_images'] > 0) & (class_summary['total_images'] < 300)]
print('Missing classes:', missing['class'].tolist())
print('Low-count under 300:', low[['class','total_images']].to_dict('records'))

assert len(bad_labels) == 0, 'Fix bad labels before training.'

In [ ]:
# Cell 4: Exact GALVAT weighted metric definitions
# Primary metric. mAP is secondary.
#
# This is the same algorithm as workbench_platform/evals/galvat_weighted_metric.py:
# - use false negative rate = 1 - recall
# - use false discovery rate = 1 - precision
# - compute remaining weighted area under FN/FP consequence curves
# - overall score = total remaining weighted area / total possible weighted area
#
# Trial 7 has a few renamed/split classes. Those are mapped explicitly below
# by copying the closest canonical Trial 5/6 GALVAT ratings. The formula is unchanged.

from dataclasses import dataclass
from typing import Mapping, Sequence

THRESHOLDS = np.array([0.00, 0.05, 0.10, 0.20, 0.50], dtype=float)

GALVAT_RATINGS = {
    "bathroom_sign": {"fn": [8, 8, 8, 8, 9], "fp": [7, 8, 8, 8, 9]},
    "canned_good": {"fn": [2, 2, 2, 3, 4], "fp": [2, 2, 2, 3, 5]},
    "cellphone": {"fn": [10, 10, 10, 10, 10], "fp": [9, 9, 9, 9, 9]},
    "chair": {"fn": [8, 8, 8, 9, 9], "fp": [10, 10, 10, 10, 10]},
    "coffee_mug": {"fn": [8, 9, 9, 10, 10], "fp": [5, 5, 5, 5, 6]},
    "curb": {"fn": [10, 10, 10, 10, 10], "fp": [8, 8, 8, 9, 10]},
    "deodorant": {"fn": [6, 6, 7, 7, 8], "fp": [6, 6, 7, 7, 7]},
    "door": {"fn": [9, 9, 9, 9, 9], "fp": [10, 10, 10, 10, 10]},
    "drinking_glass": {"fn": [8, 8, 9, 9, 10], "fp": [5, 5, 5, 5, 6]},
    "electrical_outlet": {"fn": [7, 7, 8, 8, 9], "fp": [7, 7, 8, 8, 9]},
    "exit_sign": {"fn": [7, 7, 7, 8, 8], "fp": [7, 8, 8, 8, 9]},
    "eyeglasses": {"fn": [8, 8, 8, 9, 10], "fp": [3, 3, 4, 5, 6]},
    "fork": {"fn": [7, 7, 7, 8, 9], "fp": [6, 7, 7, 8, 8]},
    "handbag": {"fn": [9, 9, 9, 9, 9], "fp": [10, 10, 10, 10, 10]},
    "handrail": {"fn": [8, 8, 8, 9, 9], "fp": [7, 7, 8, 8, 9]},
    "hearing_aid": {"fn": [9, 9, 9, 10, 10], "fp": [4, 4, 5, 6, 7]},
    "jug": {"fn": [6, 6, 6, 7, 8], "fp": [5, 5, 6, 6, 7]},
    "key": {"fn": [10, 10, 10, 10, 10], "fp": [8, 8, 8, 8, 8]},
    "knife": {"fn": [10, 10, 10, 10, 10], "fp": [6, 7, 7, 8, 8]},
    "lemon": {"fn": [4, 4, 5, 6, 7], "fp": [4, 4, 5, 6, 7]},
    "light_switch": {"fn": [7, 7, 8, 8, 9], "fp": [5, 5, 5, 5, 6]},
    "measuring_cup": {"fn": [4, 4, 5, 6, 7], "fp": [3, 3, 3, 3, 4]},
    "medicine": {"fn": [10, 10, 10, 10, 10], "fp": [10, 10, 10, 10, 10]},
    "paper_towel_dispenser": {"fn": [5, 5, 6, 7, 7], "fp": [5, 5, 6, 7, 7]},
    "pot": {"fn": [7, 7, 8, 8, 9], "fp": [5, 5, 5, 5, 6]},
    "razor": {"fn": [8, 8, 8, 9, 9], "fp": [6, 7, 7, 8, 8]},
    "remote_control": {"fn": [4, 4, 5, 6, 7], "fp": [3, 3, 3, 4, 5]},
    "salt_pepper_shaker": {"fn": [6, 6, 6, 7, 8], "fp": [2, 2, 2, 3, 4]},
    "soap_dispenser": {"fn": [6, 6, 7, 7, 8], "fp": [7, 7, 7, 8, 8]},
    "spoon": {"fn": [5, 5, 5, 5, 6], "fp": [5, 5, 5, 5, 6]},
    "stairs": {"fn": [10, 10, 10, 10, 10], "fp": [8, 8, 8, 9, 10]},
    "table": {"fn": [7, 7, 7, 8, 8], "fp": [7, 7, 7, 8, 8]},
    "thermostat": {"fn": [2, 2, 2, 2, 3], "fp": [2, 2, 2, 2, 3]},
    "toilet": {"fn": [9, 9, 9, 10, 10], "fp": [10, 10, 10, 10, 10]},
    "toothbrush": {"fn": [6, 6, 7, 7, 8], "fp": [7, 7, 8, 8, 9]},
    "toothpaste": {"fn": [8, 8, 9, 9, 9], "fp": [8, 8, 9, 9, 9]},
    "trash_can": {"fn": [5, 5, 5, 5, 6], "fp": [6, 6, 7, 7, 7]},
    "tupperware": {"fn": [3, 3, 3, 4, 5], "fp": [2, 3, 3, 4, 5]},
    "urinal": {"fn": [9, 9, 9, 10, 10], "fp": [10, 10, 10, 10, 10]},
    "wallet": {"fn": [9, 9, 9, 9, 9], "fp": [10, 10, 10, 10, 10]},
    "white_cane": {"fn": [10, 10, 10, 10, 10], "fp": [8, 8, 8, 8, 8]},
    "window": {"fn": [4, 4, 4, 5, 5], "fp": [7, 8, 8, 8, 8]},
    "wine_glass": {"fn": [8, 8, 9, 9, 10], "fp": [5, 5, 5, 5, 6]},
    "writing_device": {"fn": [2, 2, 3, 4, 4], "fp": [1, 2, 3, 3, 4]},
}

# Trial 7 class-name compatibility. These aliases preserve the exact formula and copy
# the intended canonical consequence curve for renamed/split ontology labels.
GALVAT_RATINGS.update({
    "keys": GALVAT_RATINGS["key"],
    "medicine_bottle": GALVAT_RATINGS["medicine"],
    "cane": GALVAT_RATINGS["white_cane"],
    "tv_remote": GALVAT_RATINGS["remote_control"],
    "purse": GALVAT_RATINGS["handbag"],
    "railing": GALVAT_RATINGS["handrail"],
    "step_curb": GALVAT_RATINGS["curb"],
    "pan": GALVAT_RATINGS["pot"],
    "door_frame": GALVAT_RATINGS["door"],
})

@dataclass(frozen=True)
class GalvatScoreResult:
    overall_score: float
    per_class: pd.DataFrame


def validate_ratings(class_names: Sequence[str], ratings: Mapping[str, Mapping[str, Sequence[float]]]) -> None:
    missing = sorted(set(class_names) - set(ratings))
    if missing:
        raise ValueError(f"Missing GALVAT ratings for classes: {missing}")
    malformed = []
    for class_name, sides in ratings.items():
        if "fn" not in sides or "fp" not in sides:
            malformed.append(class_name)
            continue
        if len(sides["fn"]) != len(THRESHOLDS) or len(sides["fp"]) != len(THRESHOLDS):
            malformed.append(class_name)
    if malformed:
        raise ValueError(f"Malformed GALVAT ratings for classes: {sorted(malformed)}")


def _auc(x: Sequence[float], y: Sequence[float]) -> float:
    y_arr = np.asarray(y, dtype=float)
    x_arr = np.asarray(x, dtype=float)
    trapezoid = getattr(np, "trapezoid", np.trapz)
    return float(trapezoid(y_arr, x_arr))


def remaining_area_score(error_rate: float, ratings: Sequence[float]) -> tuple[float, float, float]:
    y = np.asarray(ratings, dtype=float)
    possible = _auc(THRESHOLDS, y)
    error = max(0.0, min(float(error_rate), 1.0))
    if error >= 0.50:
        return 0.0, possible, 0.0
    xs = np.asarray([error, *[t for t in THRESHOLDS if t > error]], dtype=float)
    ys = np.interp(xs, THRESHOLDS, y)
    remaining = _auc(xs, ys)
    normalized = remaining / possible if possible > 0 else 0.0
    return remaining, possible, normalized


def _optional_float(row: pd.Series, column: str) -> float:
    if column not in row or pd.isna(row[column]):
        return float("nan")
    return float(row[column])


def galvat_weighted_score_from_df(metrics_df: pd.DataFrame, ratings=None) -> GalvatScoreResult:
    ratings = ratings or GALVAT_RATINGS
    required = {"class", "precision", "recall"}
    missing_cols = required - set(metrics_df.columns)
    if missing_cols:
        raise ValueError(f"metrics_df is missing required columns: {sorted(missing_cols)}")
    class_names = [str(name) for name in metrics_df["class"].tolist()]
    validate_ratings(class_names, ratings)

    rows = []
    total_remaining = 0.0
    total_possible = 0.0
    for _, row in metrics_df.iterrows():
        class_name = str(row["class"])
        precision = 0.0 if pd.isna(row["precision"]) else float(row["precision"])
        recall = 0.0 if pd.isna(row["recall"]) else float(row["recall"])
        precision = max(0.0, min(precision, 1.0))
        recall = max(0.0, min(recall, 1.0))
        false_discovery_rate = 1.0 - precision
        false_negative_rate = 1.0 - recall
        fn_remaining, fn_possible, fn_score = remaining_area_score(false_negative_rate, ratings[class_name]["fn"])
        fp_remaining, fp_possible, fp_score = remaining_area_score(false_discovery_rate, ratings[class_name]["fp"])
        class_remaining = fn_remaining + fp_remaining
        class_possible = fn_possible + fp_possible
        class_score = class_remaining / class_possible if class_possible > 0 else 0.0
        total_remaining += class_remaining
        total_possible += class_possible
        rows.append({
            "class": class_name,
            "precision": precision,
            "recall": recall,
            "map50": _optional_float(row, "map50"),
            "map50_95": _optional_float(row, "map50_95"),
            "false_discovery_rate": false_discovery_rate,
            "false_positive_rate": false_discovery_rate,
            "false_negative_rate": false_negative_rate,
            "fn_score": fn_score,
            "fp_score": fp_score,
            "class_score": class_score,
            "class_galvat_score": class_score,
            "possible_weight": class_possible,
            "earned_weighted_points": class_remaining,
            "lost_weighted_points": class_possible - class_remaining,
        })
    score = total_remaining / total_possible if total_possible > 0 else 0.0
    per_class = pd.DataFrame(rows).sort_values("class_score", ascending=True)
    return GalvatScoreResult(overall_score=score, per_class=per_class)


def compute_galvat_from_per_class(df):
    result = galvat_weighted_score_from_df(df, GALVAT_RATINGS)
    return result.overall_score, result.per_class

validate_ratings(NAMES, GALVAT_RATINGS)
print('Exact GALVAT metric loaded for', len(NAMES), 'dataset classes.')
print('Using Trial 7 compatibility aliases for:', ['keys', 'medicine_bottle', 'cane', 'tv_remote', 'purse', 'railing', 'step_curb', 'pan'])



In [ ]:
# Cell 5: Train YOLO26s on A100

RUN_STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME = f'galvat_trial7_yolo26s_clean640_{RUN_STAMP}'
PROJECT_DIR = Path('/content/drive/MyDrive/galvat_trial7_training_runs')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Use YOLO26s pretrained. This matches the Trial 5/6 model family but starts clean for the new ontology/order.
MODEL_INIT = 'yolo26s.pt'
model = YOLO(MODEL_INIT)

train_args = dict(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=100,
    batch=32,
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    patience=20,
    seed=42,
    deterministic=True,
    pretrained=True,
    freeze=0,
    cos_lr=True,
    mosaic=0.25,
    close_mosaic=15,
    erasing=0.0,
    auto_augment=None,
    fliplr=0.0,
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    scale=0.25,
    translate=0.05,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    amp=True,
)

print('Run name:', RUN_NAME)
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

results = model.train(**train_args)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

assert BEST_PT.exists(), f'Missing best checkpoint: {BEST_PT}'
print('RUN_DIR:', RUN_DIR)
print('BEST_PT:', BEST_PT)
print('LAST_PT:', LAST_PT)

In [ ]:
def metric_array_value(box, attr, class_id, num_classes):
    arr = np.asarray(getattr(box, attr, []), dtype=float)

    if len(arr) == num_classes:
        return float(arr[class_id])

    present = getattr(box, "ap_class_index", None)
    if present is None:
        present = getattr(box, "ap_class_indices", None)

    if present is None:
        return 0.0

    present = np.asarray(present, dtype=int).tolist()
    if class_id not in present:
        return 0.0

    pos = present.index(class_id)
    return float(arr[pos]) if pos < len(arr) else 0.0

def ultralytics_per_class_df(metrics, names, split):
    box = metrics.box
    rows = []

    support = class_summary.set_index("class").to_dict()
    img_col = f"{split}_images"
    box_col = f"{split}_boxes"

    for class_id, cls in enumerate(names):
        rows.append({
            "class_id": class_id,
            "class": cls,
            "precision": metric_array_value(box, "p", class_id, len(names)),
            "recall": metric_array_value(box, "r", class_id, len(names)),
            "map50": metric_array_value(box, "ap50", class_id, len(names)),
            "map50_95": metric_array_value(box, "ap", class_id, len(names)),
            img_col: int(support.get(img_col, {}).get(cls, 0)),
            box_col: int(support.get(box_col, {}).get(cls, 0)),
        })

    return pd.DataFrame(rows)

def compute_full_and_supported_galvat(per_class_df, split):
    full_score, full_pc = compute_galvat_from_per_class(per_class_df)

    support_col = f"{split}_images"
    supported_df = per_class_df[per_class_df[support_col] > 0].copy()

    supported_score, supported_pc = compute_galvat_from_per_class(supported_df)

    return {
        "full_score": full_score,
        "full_per_class": full_pc,
        "supported_score": supported_score,
        "supported_per_class": supported_pc,
        "supported_class_count": int(len(supported_df)),
        "zero_support_class_count": int((per_class_df[support_col] == 0).sum()),
    }

VAL_SWEEP_DIR = RUN_DIR / "galvat_threshold_sweep_val_corrected"
VAL_SWEEP_DIR.mkdir(parents=True, exist_ok=True)

CONF_VALUES = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]
IOU_VALUES = [0.45, 0.55, 0.65, 0.70]

sweep_model = YOLO(str(BEST_PT))
summary_rows = []
all_class_rows = []

for conf in CONF_VALUES:
    for iou in IOU_VALUES:
        run_label = f"val_conf{str(conf).replace('.', 'p')}_iou{str(iou).replace('.', 'p')}"
        m = sweep_model.val(
            data=str(DATA_YAML),
            split="val",
            imgsz=640,
            conf=conf,
            iou=iou,
            plots=False,
            save_json=False,
            project=str(VAL_SWEEP_DIR),
            name=run_label,
            exist_ok=True,
            verbose=False,
        )

        pc = ultralytics_per_class_df(m, NAMES, "valid")
        scores = compute_full_and_supported_galvat(pc, "valid")

        supported_pc = scores["supported_per_class"].copy()
        supported_pc["conf"] = conf
        supported_pc["iou"] = iou
        all_class_rows.append(supported_pc)

        summary_rows.append({
            "split": "val",
            "conf": conf,
            "iou": iou,
            "galvat_supported_score": scores["supported_score"],
            "galvat_full_ontology_score": scores["full_score"],
            "supported_class_count": scores["supported_class_count"],
            "zero_support_class_count": scores["zero_support_class_count"],
            "precision_macro_supported": float(pc[pc["valid_images"] > 0]["precision"].mean()),
            "recall_macro_supported": float(pc[pc["valid_images"] > 0]["recall"].mean()),
            "map50_macro_supported": float(pc[pc["valid_images"] > 0]["map50"].mean()),
            "map50_95_macro_supported": float(pc[pc["valid_images"] > 0]["map50_95"].mean()),
            "zero_galvat_supported_classes": int((supported_pc["class_score"] <= 0).sum()),
            "supported_classes_below_0_2": int((supported_pc["class_score"] < 0.2).sum()),
        })

        print(run_label, "supported GALVAT=", round(scores["supported_score"], 4))

val_summary = pd.DataFrame(summary_rows).sort_values("galvat_supported_score", ascending=False)
val_per_class = pd.concat(all_class_rows, ignore_index=True)

val_summary.to_csv(VAL_SWEEP_DIR / "val_threshold_sweep_summary_corrected.csv", index=False)
val_per_class.to_csv(VAL_SWEEP_DIR / "val_threshold_sweep_supported_per_class_corrected.csv", index=False)

BEST_ROW = val_summary.iloc[0].to_dict()
SELECTED_CONF = float(BEST_ROW["conf"])
SELECTED_IOU = float(BEST_ROW["iou"])

print("Selected by corrected validation supported GALVAT:")
print(BEST_ROW)
display(val_summary)

In [ ]:
TEST_DIR = RUN_DIR / "selected_threshold_test_eval_corrected"
TEST_DIR.mkdir(parents=True, exist_ok=True)

def compute_well_represented_galvat(per_class_df, split, min_images=30):
    support_col = f"{split}_images"
    well_df = per_class_df[per_class_df[support_col] >= min_images].copy()
    score, pc = compute_galvat_from_per_class(well_df)
    return score, pc, len(well_df)

test_model = YOLO(str(BEST_PT))
test_metrics = test_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    conf=SELECTED_CONF,
    iou=SELECTED_IOU,
    plots=True,
    save_json=True,
    project=str(TEST_DIR),
    name="test_selected_corrected",
    exist_ok=True,
    verbose=False,
)

test_pc = ultralytics_per_class_df(test_metrics, NAMES, "test")

test_scores = compute_full_and_supported_galvat(test_pc, "test")
test_galvat = test_scores["supported_score"]
test_galvat_pc = test_scores["supported_per_class"]

test_well_galvat, test_well_pc, test_well_count = compute_well_represented_galvat(
    test_pc,
    "test",
    min_images=30,
)

test_pc.to_csv(TEST_DIR / "test_selected_per_class_metrics_corrected.csv", index=False)
test_galvat_pc.to_csv(TEST_DIR / "test_selected_supported_galvat_scores.csv", index=False)
test_well_pc.to_csv(TEST_DIR / "test_selected_well_represented_galvat_scores.csv", index=False)

supported_test = test_pc[test_pc["test_images"] > 0]
well_test = test_pc[test_pc["test_images"] >= 30]

test_summary = {
    "split": "test",
    "conf": SELECTED_CONF,
    "iou": SELECTED_IOU,

    "galvat_supported_score": test_galvat,
    "galvat_full_ontology_score": test_scores["full_score"],
    "galvat_well_represented_score_min30": test_well_galvat,

    "supported_class_count": test_scores["supported_class_count"],
    "well_represented_class_count_min30": test_well_count,
    "zero_support_class_count": test_scores["zero_support_class_count"],

    "precision_macro_supported": float(supported_test["precision"].mean()),
    "recall_macro_supported": float(supported_test["recall"].mean()),
    "map50_macro_supported": float(supported_test["map50"].mean()),
    "map50_95_macro_supported": float(supported_test["map50_95"].mean()),

    "precision_macro_well_represented": float(well_test["precision"].mean()),
    "recall_macro_well_represented": float(well_test["recall"].mean()),
    "map50_macro_well_represented": float(well_test["map50"].mean()),
    "map50_95_macro_well_represented": float(well_test["map50_95"].mean()),

    "zero_galvat_supported_classes": int((test_galvat_pc["class_score"] <= 0).sum()),
    "supported_classes_below_0_2": int((test_galvat_pc["class_score"] < 0.2).sum()),

    "zero_galvat_well_represented_classes": int((test_well_pc["class_score"] <= 0).sum()),
    "well_represented_classes_below_0_2": int((test_well_pc["class_score"] < 0.2).sum()),
}

(TEST_DIR / "test_selected_summary_corrected.json").write_text(json.dumps(test_summary, indent=2))

print(json.dumps(test_summary, indent=2))
print("\nWeakest supported classes:")
display(test_galvat_pc.sort_values("class_score").head(20))

print("\nWeakest well-represented classes:")
display(test_well_pc.sort_values("class_score").head(20))

In [ ]:
# Cell 8: Background FP and clutter/spoken-burden evaluation

ROI_DIR = RUN_DIR / 'roi_fp_spoken_eval'
ROI_DIR.mkdir(parents=True, exist_ok=True)

eval_model = YOLO(str(BEST_PT))

def safe_div(a, b):
      return float(a) / float(b) if b else 0.0

def label_file_for_image(img_path):
    return Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')

def count_labels(img_path):
    lbl = label_file_for_image(img_path)
    if not lbl.exists():
        return 0
    return sum(1 for x in lbl.read_text().splitlines() if x.strip())

def sample_images(split, predicate, n, seed=42):
    rng = random.Random(seed)
    img_dir = DATA_ROOT / split / 'images'
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS and predicate(p)]
    rng.shuffle(imgs)
    return imgs[:n]

def predict_image_list(image_paths, out_name, conf, iou):
    pred_rows = []
    if not image_paths:
        return pd.DataFrame()
    results = eval_model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, iou=iou, verbose=False, stream=True)
    for img_path, res in zip(image_paths, results):
        boxes = res.boxes
        if boxes is None or len(boxes) == 0:
            pred_rows.append({'image': Path(img_path).name, 'class': None, 'conf': None})
            continue
        for b in boxes:
            cls_id = int(b.cls.item())
            pred_rows.append({'image': Path(img_path).name, 'class': NAMES[cls_id], 'conf': float(b.conf.item())})
    df = pd.DataFrame(pred_rows)
    df.to_csv(ROI_DIR / f'{out_name}_predictions.csv', index=False)
    return df

bg_imgs = sample_images('test', lambda p: count_labels(p) == 0, 600, seed=43)
if len(bg_imgs) < 100:
    # Fall back to validation/train empty-label images if test has too few.
    bg_imgs += sample_images('valid', lambda p: count_labels(p) == 0, 600 - len(bg_imgs), seed=44)
    bg_imgs += sample_images('train', lambda p: count_labels(p) == 0, 600 - len(bg_imgs), seed=45)
bg_imgs = bg_imgs[:600]

clutter_imgs = sample_images('test', lambda p: count_labels(p) >= 4, 500, seed=46)
if len(clutter_imgs) < 100:
    clutter_imgs += sample_images('valid', lambda p: count_labels(p) >= 4, 500 - len(clutter_imgs), seed=47)
clutter_imgs = clutter_imgs[:500]

bg_pred = predict_image_list(bg_imgs, 'background_empty', SELECTED_CONF, SELECTED_IOU)
clutter_pred = predict_image_list(clutter_imgs, 'cluttered', SELECTED_CONF, SELECTED_IOU)

def summarize_predictions(df, image_paths, name):
    total = len(image_paths)
    if total == 0:
        return {'name': name, 'roi_count': 0}
    if df.empty:
        det_counts = pd.Series(dtype=int)
        high_conf_count = 0
    else:
        real = df[df['class'].notna()].copy()
        det_counts = real.groupby('image').size()
        high_conf_count = int((real['conf'] >= 0.75).sum()) if len(real) else 0
    return {
        'name': name,
        'roi_count': total,
        'rois_with_any_detection': int((det_counts > 0).sum()),
        'roi_any_detection_rate': safe_div(int((det_counts > 0).sum()), total),
        'mean_detections_per_roi': float(det_counts.sum() / total) if total else 0.0,
        'max_detections_per_roi': int(det_counts.max()) if len(det_counts) else 0,
        'high_conf_detection_count': high_conf_count,
    }

bg_summary = summarize_predictions(bg_pred, bg_imgs, 'background_empty')
clutter_summary = summarize_predictions(clutter_pred, clutter_imgs, 'cluttered')

spoken_budget = 3
if not clutter_pred.empty:
    real = clutter_pred[clutter_pred['class'].notna()].copy()
    counts = real.groupby('image').size() if len(real) else pd.Series(dtype=int)
    over_budget = int((counts > spoken_budget).sum())
else:
    over_budget = 0
clutter_summary['spoken_budget_detections_per_roi'] = spoken_budget
clutter_summary['over_budget_rate'] = safe_div(over_budget, len(clutter_imgs))

roi_summary = {'conf': SELECTED_CONF, 'iou': SELECTED_IOU, 'background': bg_summary, 'clutter': clutter_summary}
(ROI_DIR / 'roi_fp_spoken_summary.json').write_text(json.dumps(roi_summary, indent=2))
print(json.dumps(roi_summary, indent=2))

In [ ]:
# Cell 8B: Fresh non-overlap background ROI false-positive evaluation

FRESH_BG_DIR = RUN_DIR / "fresh_background_roi_eval"
FRESH_BG_IMG_DIR = FRESH_BG_DIR / "images"
FRESH_BG_IMG_DIR.mkdir(parents=True, exist_ok=True)

def safe_div(a, b):
      return float(a) / float(b) if b else 0.0

def read_yolo_boxes_abs(img_path):
    lbl_path = label_file_for_image(img_path)
    if not lbl_path.exists():
        return []

    im = Image.open(img_path)
    W, H = im.size

    boxes = []
    for line in lbl_path.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        if len(parts) != 5:
            continue

        _, xc, yc, w, h = [float(x) for x in parts]
        x1 = (xc - w / 2) * W
        y1 = (yc - h / 2) * H
        x2 = (xc + w / 2) * W
        y2 = (yc + h / 2) * H
        boxes.append((x1, y1, x2, y2))

    return boxes

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def crop_touches_any_box(crop_box, gt_boxes, max_iou=0.02):
    for gt in gt_boxes:
        if iou_xyxy(crop_box, gt) > max_iou:
            return True
    return False

def make_fresh_background_crops(
    split,
    target_crops=300,
    crop_size=320,
    max_attempts_per_image=25,
    seed=42,
):
    rng = random.Random(seed)

    img_dir = DATA_ROOT / split / "images"
    candidates = [
        p for p in img_dir.glob("*")
        if p.suffix.lower() in IMAGE_EXTS and count_labels(p) > 0
    ]
    rng.shuffle(candidates)

    rows = []
    made = 0

    for img_path in tqdm(candidates, desc=f"Making fresh background crops from {split}"):
        if made >= target_crops:
            break

        try:
            im = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        W, H = im.size
        if W < crop_size or H < crop_size:
            continue

        gt_boxes = read_yolo_boxes_abs(img_path)
        if not gt_boxes:
            continue

        for _ in range(max_attempts_per_image):
            if made >= target_crops:
                break

            x1 = rng.randint(0, W - crop_size)
            y1 = rng.randint(0, H - crop_size)
            x2 = x1 + crop_size
            y2 = y1 + crop_size
            crop_box = (x1, y1, x2, y2)

            if crop_touches_any_box(crop_box, gt_boxes, max_iou=0.02):
                continue

            crop = im.crop(crop_box).resize((640, 640), Image.Resampling.LANCZOS)
            out_name = f"fresh_bg_{split}_{made:04d}_{img_path.stem}.jpg"
            out_path = FRESH_BG_IMG_DIR / out_name
            crop.save(out_path, quality=92)

            rows.append({
                "split": split,
                "source_image": str(img_path),
                "crop_image": str(out_path),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "crop_size": crop_size,
            })

            made += 1

    return pd.DataFrame(rows)

fresh_valid = make_fresh_background_crops("valid", target_crops=300, seed=100)
fresh_test = make_fresh_background_crops("test", target_crops=300, seed=200)

fresh_bg_meta = pd.concat([fresh_valid, fresh_test], ignore_index=True)
fresh_bg_meta.to_csv(FRESH_BG_DIR / "fresh_background_roi_metadata.csv", index=False)

fresh_bg_paths = [Path(p) for p in fresh_bg_meta["crop_image"].tolist()]

fresh_bg_pred = predict_image_list(
    fresh_bg_paths,
    "fresh_background_nonoverlap",
    SELECTED_CONF,
    SELECTED_IOU,
)

fresh_bg_summary = summarize_predictions(
    fresh_bg_pred,
    fresh_bg_paths,
    "fresh_background_nonoverlap",
)

fresh_bg_summary["crop_count_valid"] = int(len(fresh_valid))
fresh_bg_summary["crop_count_test"] = int(len(fresh_test))
fresh_bg_summary["crop_generation"] = "random 320x320 non-overlap crops from labeled valid/test images, resized to 640"
fresh_bg_summary["max_allowed_crop_gt_iou"] = 0.02

(FRESH_BG_DIR / "fresh_background_roi_summary.json").write_text(
    json.dumps(fresh_bg_summary, indent=2)
)

print(json.dumps(fresh_bg_summary, indent=2))
display(fresh_bg_meta.head())
display(fresh_bg_pred.head() if len(fresh_bg_pred) else fresh_bg_pred)

In [ ]:
# Corrected Cell 9: Graphs, charts, and boss-readable tables

CHART_DIR = RUN_DIR / "charts_corrected"
CHART_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

plt.figure(figsize=(9, 5))
sns.lineplot(data=val_summary, x="conf", y="galvat_supported_score", hue="iou", marker="o")
plt.axvline(SELECTED_CONF, color="red", linestyle="--", label=f"selected conf={SELECTED_CONF}")
plt.title("Validation Supported GALVAT Score by Confidence and IoU")
plt.ylabel("supported GALVAT score")
plt.tight_layout()
plt.savefig(CHART_DIR / "01_val_supported_galvat_by_conf_iou.png", dpi=180)
plt.show()

pivot = val_summary.pivot(index="iou", columns="conf", values="galvat_supported_score")
plt.figure(figsize=(9, 4.5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.title("Validation Supported GALVAT Heatmap")
plt.tight_layout()
plt.savefig(CHART_DIR / "02_val_supported_galvat_heatmap.png", dpi=180)
plt.show()

plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=val_summary,
    x="recall_macro_supported",
    y="precision_macro_supported",
    size="galvat_supported_score",
    hue="conf",
    palette="magma",
    sizes=(80, 350),
)
plt.title("Validation Supported Precision/Recall Tradeoff")
plt.tight_layout()
plt.savefig(CHART_DIR / "03_supported_precision_recall_tradeoff.png", dpi=180)
plt.show()

weak_test = test_galvat_pc.sort_values("class_score").head(20)
plt.figure(figsize=(10, 7))
sns.barplot(data=weak_test, y="class", x="class_score", color="firebrick")
plt.title("20 Weakest Supported Test Classes by GALVAT Class Score")
plt.xlabel("class GALVAT score")
plt.tight_layout()
plt.savefig(CHART_DIR / "04_weakest_supported_test_classes.png", dpi=180)
plt.show()

well_test_plot = test_well_pc.sort_values("class_score").head(20)
plt.figure(figsize=(10, 7))
sns.barplot(data=well_test_plot, y="class", x="class_score", color="darkorange")
plt.title("Weakest Well-Represented Test Classes by GALVAT Class Score")
plt.xlabel("class GALVAT score")
plt.tight_layout()
plt.savefig(CHART_DIR / "05_weakest_well_represented_test_classes.png", dpi=180)
plt.show()

roi_bar = pd.DataFrame([
    {"metric": "background_any_detection_rate", "value": bg_summary.get("roi_any_detection_rate", 0)},
    {"metric": "fresh_background_any_detection_rate", "value": fresh_bg_summary.get("roi_any_detection_rate", 0) if "fresh_bg_summary"
    in globals() else None},
    {"metric": "clutter_over_budget_rate", "value": clutter_summary.get("over_budget_rate", 0)},
]).dropna()

plt.figure(figsize=(8, 4))
sns.barplot(data=roi_bar, x="metric", y="value", color="steelblue")
plt.ylim(0, max(0.1, float(roi_bar["value"].max()) * 1.25))
plt.title("Assistive-Use ROI Risk Checks")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "06_roi_risk_summary.png", dpi=180)
plt.show()

slide_table = pd.DataFrame([
    {"item": "selected_conf", "value": SELECTED_CONF},
    {"item": "selected_iou", "value": SELECTED_IOU},
    {"item": "val_supported_galvat", "value": BEST_ROW["galvat_supported_score"]},
    {"item": "val_full_ontology_galvat", "value": BEST_ROW["galvat_full_ontology_score"]},
    {"item": "test_supported_galvat", "value": test_summary["galvat_supported_score"]},
    {"item": "test_full_ontology_galvat", "value": test_summary["galvat_full_ontology_score"]},
    {"item": "test_well_represented_galvat_min30", "value": test_summary["galvat_well_represented_score_min30"]},
    {"item": "supported_class_count", "value": test_summary["supported_class_count"]},
    {"item": "well_represented_class_count_min30", "value": test_summary["well_represented_class_count_min30"]},
    {"item": "zero_support_class_count", "value": test_summary["zero_support_class_count"]},
    {"item": "background_any_detection_rate", "value": bg_summary.get("roi_any_detection_rate", None)},
    {"item": "fresh_background_any_detection_rate", "value": fresh_bg_summary.get("roi_any_detection_rate", None) if "fresh_bg_summary"
    in globals() else None},
    {"item": "clutter_over_budget_rate", "value": clutter_summary.get("over_budget_rate", None)},
])

slide_table.to_csv(CHART_DIR / "boss_summary_table_corrected.csv", index=False)
display(slide_table)

In [ ]:
# Cell 10: Export ONNX and validate artifact exists

export_model = YOLO(str(BEST_PT))
onnx_path = export_model.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
onnx_path = Path(onnx_path)
assert onnx_path.exists(), f'ONNX export failed: {onnx_path}'
print('ONNX exported:', onnx_path)

# Copy ONNX into run root if Ultralytics saved elsewhere.
target_onnx = RUN_DIR / onnx_path.name
if onnx_path.resolve() != target_onnx.resolve():
    shutil.copy2(onnx_path, target_onnx)
print('ONNX in run dir:', target_onnx)

In [ ]:
# Cell 11: Final run summary and archive folder in Drive

ARCHIVE_DIR = Path('/content/drive/MyDrive/galvat_trial7_final_model_outputs') / RUN_NAME
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

final_summary = {
    "run_name": RUN_NAME,
    "created_at": RUN_STAMP,
    "dataset_zip": str(DATASET_ZIP),
    "data_yaml": str(DATA_YAML),
    "model_init": MODEL_INIT,
    "best_pt": str(BEST_PT),
    "last_pt": str(LAST_PT),
    "onnx": str(target_onnx),

    "selected_conf": SELECTED_CONF,
    "selected_iou": SELECTED_IOU,

    "val_supported_galvat": float(BEST_ROW["galvat_supported_score"]),
    "val_full_ontology_galvat": float(BEST_ROW["galvat_full_ontology_score"]),

    "test_supported_galvat": float(test_summary["galvat_supported_score"]),
    "test_full_ontology_galvat": float(test_summary["galvat_full_ontology_score"]),
    "test_well_represented_galvat_min30": float(test_summary["galvat_well_represented_score_min30"]),

    "supported_class_count": int(test_summary["supported_class_count"]),
    "well_represented_class_count_min30": int(test_summary["well_represented_class_count_min30"]),
    "zero_support_class_count": int(test_summary["zero_support_class_count"]),

    "background_any_detection_rate": bg_summary.get("roi_any_detection_rate"),
    "fresh_background_any_detection_rate": fresh_bg_summary.get("roi_any_detection_rate") if "fresh_bg_summary" in globals() else None,
    "clutter_over_budget_rate": clutter_summary.get("over_budget_rate"),

    "primary_metric": "GALVAT weighted score",
    "note": "Threshold selected on validation supported GALVAT. Full ontology GALVAT includes missing dataset classes. mAP issecondary.",
}

(RUN_DIR / 'trial7_final_run_summary.json').write_text(json.dumps(final_summary, indent=2))

# Copy the entire Ultralytics run directory into final archive folder.
shutil.copytree(RUN_DIR, ARCHIVE_DIR / 'training_run', dirs_exist_ok=True)

# Copy important top-level files for easy access.
for src in [BEST_PT, LAST_PT, target_onnx, DATA_YAML, RUN_DIR / 'trial7_final_run_summary.json']:
    if Path(src).exists():
        shutil.copy2(src, ARCHIVE_DIR / Path(src).name)

split_summary.to_csv(ARCHIVE_DIR / 'dataset_split_summary.csv', index=False)
class_summary.to_csv(ARCHIVE_DIR / 'dataset_class_summary.csv', index=False)

print('Final archive saved to:')
print(ARCHIVE_DIR)
print('\nKey files:')
for p in sorted(ARCHIVE_DIR.iterdir()):
    print('-', p.name)

print('\nFinal summary:')
print(json.dumps(final_summary, indent=2))